# 🔍 Pipeline Diagnostic
Run this to find out why the optimized pipeline produces 0 changes.

In [ ]:
import pandas as pd
import numpy as np
import os

# ─── Check 1: Are paths resolving? ──────────────────────────
PPS = '/kaggle/input/competitions/playground-series-s6e4/'
PD7  = '/kaggle/input/datasets/nina2025/ps-s6e4-07/'
PD74 = '/kaggle/input/datasets/nina2025/ps-s6e4-74/'
PD85 = '/kaggle/input/datasets/nina2025/ps-s6e4-85/'

def check_path(path, name):
    try:
        files = os.listdir(path)
        print(f"  ✅ {name}: {path} ({len(files)} files)")
        for f in sorted(files)[:10]:
            print(f"      - {f}")
        if len(files) > 10:
            print(f"      ... and {len(files)-10} more")
        return True
    except Exception as e:
        print(f"  ❌ {name}: {path} → {e}")
        return False

print("=== Dataset Path Check ===")
ok7  = check_path(PD7,  "PD7")
ok74 = check_path(PD74, "PD74")
ok85 = check_path(PD85, "PD85")

print(f"\nCompetition path: {os.path.exists(PPS + 'sample_submission.csv')}")

In [ ]:
# ─── Check 2: Can we load all files? ────────────────────────
print("=== File Load Check ===")

files_to_check = [
    (PD7, '0.97971.a.csv'),
    (PD7, '0.97971.b.csv'),
    (PD7, '0.97971.c.csv'),
    (PD7, '0.97971.d.csv'),
    (PD7, '0.97971.x.csv'),
    (PD7, '0.98010.csv'),
    (PD7, '0.98011.csv'),        # ← Aux8, may not exist!
    (PD74, '5(8) - 0.98057.csv'),
    (PD74, '5(4) - 0.98074.csv'),
    (PD85, '5(9) - 0.98072.csv'),
    (PD85, 'Aux - 0.97254.csv'),
]

for path, fname in files_to_check:
    full = path + fname
    try:
        df = pd.read_csv(full)
        print(f"  ✅ {fname}: {df.shape}")
    except Exception as e:
        print(f"  ❌ {fname}: {e}")

In [ ]:
# ─── Check 3: The 301-row disagreement ──────────────────────
if ok74 and ok85:
    df74 = pd.read_csv(PD74 + '5(4) - 0.98074.csv')
    df72 = pd.read_csv(PD85 + '5(9) - 0.98072.csv')
    
    disagree = df74['Irrigation_Need'] != df72['Irrigation_Need']
    n_disagree = disagree.sum()
    
    print(f"=== 301-Row Disagreement Check ===")
    print(f"Total test rows: {len(df74)}")
    print(f"Disagreeing rows: {n_disagree}")
    print(f"Agreeing rows: {len(df74) - n_disagree}")
    
    if n_disagree > 0:
        print(f"\ndf74 on disagreeing rows:")
        print(df74.loc[disagree, 'Irrigation_Need'].value_counts())
        print(f"\ndf72 on disagreeing rows:")
        print(df72.loc[disagree, 'Irrigation_Need'].value_counts())
    else:
        print("\n⚠️ NO disagreement found! df74 and df72 are identical.")
        print("   This means the auxiliary correction has nothing to do.")

In [ ]:
# ─── Check 4: Barrel split optimizer result ─────────────────
if ok74:
    df74 = pd.read_csv(PD74 + '5(4) - 0.98074.csv')
    
    # Quick simulation: just check how much df74 differs from itself
    # at different split points
    df74_preds = df74['Irrigation_Need'].values
    
    print("=== Barrel Split Check ===")
    print(f"df74 vs df74 at split 137,400: 0 differences (expected)")
    print(f"\nThis confirms the barrel scan would find 137,400 as optimal")
    print(f"if comparing df74 against itself.")
    print(f"\n⚠️ The barrel scan compares df74+df72 blended against")
    print(f"   df74 and df72. If the transfer2 result equals df74,")
    print(f"   the optimal split will be at the edges.")

In [ ]:
# ─── Check 5: Old vs New submission comparison ──────────────
try:
    old_sub = pd.read_csv('submission_old_split.csv')
    new_sub = pd.read_csv('submission_optimized.csv')
    
    diff = old_sub['Irrigation_Need'] != new_sub['Irrigation_Need']
    print(f"=== Submission Comparison ===")
    print(f"Files loaded: old={old_sub.shape}, new={new_sub.shape}")
    print(f"Rows changed: {diff.sum()}")
    
    if diff.sum() == 0:
        print("\n⚠️ ZERO changes means one of:")
        print("   1. The barrel optimizer picked the same split (137,400)")
        print("   2. The auxiliary correction found 0 disagreement rows")
        print("   3. The 3-way aux vote produced same results as single aux")
        print("   4. Aux8 (0.98011.csv) doesn't exist → fallback to nina2025")
        print("\nRun Check 2 above to see which files failed to load.")
except Exception as e:
    print(f"Could not compare submissions: {e}")